<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">
            Creación de un framework para recomendar las personas más adecuadas para trabajar en un determinado proyecto de ingeniería del software
        </h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Profesores
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barceló
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 3: Minería de Reglas de Asociación)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        El objetivo de esta libreta es <strong>descubrir patrones ocultos y relaciones entre las habilidades blandas, preferencias y características de la personalidad</strong> de los candidatos, utilizando algoritmos de aprendizaje no supervisado.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Carga y optimización:</strong> Lectura de la matriz transaccional generada en la libreta anterior y conversión a tipo booleano para maximizar la eficiencia de memoria.</li>
        <li><strong>Extracción de Itemsets Frecuentes:</strong> Aplicación del algoritmo <strong>FP-Growth</strong> (más eficiente computacionalmente que Apriori para matrices grandes) para encontrar combinaciones de características que aparecen juntas de manera representativa (Soporte).</li>
        <li><strong>Generación de Reglas:</strong> Creación de reglas direccionales (Antecedente -> Consecuente) basadas en métricas de probabilidad condicional (<em>Confianza</em>).</li>
        <li><strong>Filtrado por Relevancia:</strong> Poda de reglas triviales utilizando el <em>Lift</em> para quedarnos solo con aquellas que representen una correlación fuerte y real entre variables.</li>
        <li><strong>Exportación del modelo:</strong> Guardado de las reglas finales como "Base de Conocimiento" para cruzarla posteriormente con las reglas expertas de riesgo del framework NetSTAFF.</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 3.1. Configuración del Entorno y carga de la Matriz Transaccional

Importamos las librerías necesarias. En este caso haremos uso de `mlxtend` para la minería de reglas. Cargaremos la matriz generada previamente y la convertiremos al tipo de dato booleano (`True`/`False`), que es el formato óptimo requerido por las versiones modernas de `mlxtend` para un procesamiento rápido y de bajo consumo de memoria.

In [1]:
import os
import sys
import warnings
import time

sys.path.append(os.path.abspath(os.path.join('../..')))
from src.soft_skills import config

import pandas as pd
from mlxtend.frequent_patterns import apriori,fpgrowth, association_rules
from IPython.display import display

warnings.filterwarnings("ignore")

df_matriz = pd.read_csv(config.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId')
df_matriz = df_matriz.fillna(0).astype(bool)

print(f"Dimensiones matriz: {df_matriz.shape}")
display(df_matriz.head(3))

Dimensiones matriz: (88778, 205)


,WorkWeekHrs_is_missing,CodeRevHrs_is_missing,WorkChallenge_Distracting work environment,WorkChallenge_Meetings,WorkChallenge_Being tasked with non-development work,WorkChallenge_Not enough people for the workload,WorkChallenge_Lack of support from management,WorkChallenge_Inadequate access to necessary tools,WorkChallenge_Toxic work environment,WorkChallenge_Time spent commuting,...,Apertura_Nivel_Bajo,Apertura_Nivel_Medio,WorkWeekHrs_Bin_35-45h,WorkWeekHrs_Bin_45-55h,WorkWeekHrs_Bin_<35h,WorkWeekHrs_Bin_>55h,CodeRevHrs_Bin_0-2h,CodeRevHrs_Bin_0h (No revisa nada de código),CodeRevHrs_Bin_2-5h,CodeRevHrs_Bin_>5h
CandidateId,,,,,,,,,,,,,,,,,,,,,
1,True,True,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
2,True,True,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
3,False,True,True,False,False,False,True,True,False,False,...,False,True,True,False,False,False,False,False,False,False


### 3.2. Generación de Itemsets Frecuentes (Algoritmo Apriori)



In [2]:
itemsets_frecuentes = apriori(
    df_matriz,
    min_support=0.15,
    use_colnames=True,
    max_len=4,
    low_memory=True,
    verbose=1
)

Processing 3045 combinations | Sampling itemset size 4


In [3]:
pd.set_option('display.max_colwidth', None)
display(itemsets_frecuentes.sort_values('support', ascending=False).head(20))

,support,itemsets
18,0.802643,frozenset({Hobbyist_Yes})
22,0.725461,frozenset({Employment_Employed full-time})
65,0.705817,frozenset({WelcomeChange_Just as welcome now as I felt last year})
54,0.618824,frozenset({BetterLife_Yes})
67,0.605803,frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})
58,0.595418,frozenset({Extraversion_In real life (in person)})
256,0.572022,"frozenset({Hobbyist_Yes, Employment_Employed full-time})"
286,0.564464,"frozenset({Hobbyist_Yes, WelcomeChange_Just as welcome now as I felt last year})"
354,0.558832,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Employment_Employed full-time})"
48,0.545450,"frozenset({CodeRev_Yes, because I see value in code review})"


In [4]:
reglas_asociacion = association_rules(
    itemsets_frecuentes,
    metric="confidence",
    min_threshold=0.6
)

In [5]:
reglas_asociacion = reglas_asociacion.sort_values('lift', ascending=False)

print(f"Reglas generadas: {len(reglas_asociacion)}")
display(reglas_asociacion[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas generadas: 5324


,antecedents,consequents,support,confidence,lift
2496,"frozenset({Hobbyist_Yes, WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})",frozenset({Neuroticismo_Nivel_Desconocido}),0.154385,0.881471,4.912138
2503,frozenset({Neuroticismo_Nivel_Desconocido}),"frozenset({Hobbyist_Yes, WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})",0.154385,0.860335,4.912138
2514,frozenset({Neuroticismo_Nivel_Desconocido}),"frozenset({Hobbyist_Yes, CodeRevHrs_is_missing, WorkRemote_Bucket_Desconocido (Sin datos)})",0.154385,0.860335,4.897963
2508,"frozenset({Hobbyist_Yes, CodeRevHrs_is_missing, WorkRemote_Bucket_Desconocido (Sin datos)})",frozenset({Neuroticismo_Nivel_Desconocido}),0.154385,0.878928,4.897963
2490,frozenset({Neuroticismo_Nivel_Desconocido}),"frozenset({WorkWeekHrs_is_missing, CodeRevHrs_is_missing, WorkRemote_Bucket_Desconocido (Sin datos)})",0.179448,1.000000,4.879789
2483,"frozenset({WorkWeekHrs_is_missing, CodeRevHrs_is_missing, WorkRemote_Bucket_Desconocido (Sin datos)})",frozenset({Neuroticismo_Nivel_Desconocido}),0.179448,0.875666,4.879789
2484,"frozenset({Neuroticismo_Nivel_Desconocido, CodeRevHrs_is_missing})","frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})",0.179448,1.000000,4.862683
435,"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})",frozenset({Neuroticismo_Nivel_Desconocido}),0.179448,0.872597,4.862683
436,frozenset({Neuroticismo_Nivel_Desconocido}),"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})",0.179448,1.000000,4.862683
2489,"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})","frozenset({Neuroticismo_Nivel_Desconocido, CodeRevHrs_is_missing})",0.179448,0.872597,4.862683


In [6]:
def contiene_item_missing(frozenset_items):
    return any('is_missing' in item or 'Desconocido' in item for item in frozenset_items)

reglas_asociacion['tiene_missing'] = reglas_asociacion.apply(
    lambda row: contiene_item_missing(row['antecedents']) or contiene_item_missing(row['consequents']),
    axis=1
)

reglas_sustantivas = reglas_asociacion[~reglas_asociacion['tiene_missing']].sort_values('lift', ascending=False)

print(f"Reglas totales: {len(reglas_asociacion)}")
print(f"Reglas de estructura de nulos (excluidas del análisis sustantivo): {reglas_asociacion['tiene_missing'].sum()}")
print(f"Reglas sustantivas restantes: {len(reglas_sustantivas)}")
display(reglas_sustantivas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas totales: 5324
Reglas de estructura de nulos (excluidas del análisis sustantivo): 149
Reglas sustantivas restantes: 5175


,antecedents,consequents,support,confidence,lift
4255,"frozenset({CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})","frozenset({Amabilidad_Nivel_Alto, Employment_Employed full-time})",0.169952,0.901584,3.821839
4251,"frozenset({Amabilidad_Nivel_Alto, Employment_Employed full-time})","frozenset({CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})",0.169952,0.720432,3.821839
4266,"frozenset({Amabilidad_Nivel_Alto, WelcomeChange_Just as welcome now as I felt last year})","frozenset({Employment_Employed full-time, MgrIdiot_Very confident})",0.152842,0.964118,3.691400
2010,"frozenset({Amabilidad_Nivel_Alto, WelcomeChange_Just as welcome now as I felt last year})",frozenset({MgrIdiot_Very confident}),0.158530,1.000000,3.646812
4262,"frozenset({Amabilidad_Nivel_Alto, WelcomeChange_Just as welcome now as I felt last year, Employment_Employed full-time})",frozenset({MgrIdiot_Very confident}),0.152842,1.000000,3.646812
4227,"frozenset({MgrIdiot_Somewhat confident, WelcomeChange_Just as welcome now as I felt last year})","frozenset({Employment_Employed full-time, Amabilidad_Nivel_Medio})",0.153259,0.733042,3.403841
4229,"frozenset({Employment_Employed full-time, Amabilidad_Nivel_Medio})","frozenset({MgrIdiot_Somewhat confident, WelcomeChange_Just as welcome now as I felt last year})",0.153259,0.711648,3.403841
2006,frozenset({Amabilidad_Nivel_Alto}),"frozenset({CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})",0.176023,0.612055,3.246911
2005,"frozenset({CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})",frozenset({Amabilidad_Nivel_Alto}),0.176023,0.933791,3.246911
4250,"frozenset({Employment_Employed full-time, CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})",frozenset({Amabilidad_Nivel_Alto}),0.169952,0.933259,3.245060


In [7]:
INPUTS_BIGFIVE = {
    'Neuroticismo': ['ImpSyn', 'JobSat', 'CareerSat', 'WorkWeekHrs_Bin'],
    'Amabilidad':   ['WelcomeChange', 'CodeRev', 'MgrIdiot'],
    'Apertura':     ['Hobbyist', 'OpenSourcer', 'BetterLife'],
}

def es_regla_circular(frozenset_items):
    rasgos_presentes = set()
    inputs_presentes = set()
    for item in frozenset_items:
        for rasgo in INPUTS_BIGFIVE:
            if item.startswith(f'{rasgo}_Nivel_'):
                rasgos_presentes.add(rasgo)
        for rasgo, inputs in INPUTS_BIGFIVE.items():
            if any(item.startswith(f'{col}_') for col in inputs):
                inputs_presentes.add(rasgo)
    return len(rasgos_presentes & inputs_presentes) > 0

reglas_sustantivas['es_circular'] = reglas_sustantivas.apply(
    lambda row: es_regla_circular(row['antecedents'] | row['consequents']),
    axis=1
)

reglas_limpias = reglas_sustantivas[~reglas_sustantivas['es_circular']].sort_values('lift', ascending=False)

print(f"Reglas sustantivas (previas): {len(reglas_sustantivas)}")
print(f"Reglas circulares (proxy vs. sus propios insumos): {reglas_sustantivas['es_circular'].sum()}")
print(f"Reglas limpias restantes: {len(reglas_limpias)}")
display(reglas_limpias[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas sustantivas (previas): 5175
Reglas circulares (proxy vs. sus propios insumos): 491
Reglas limpias restantes: 4684


,antecedents,consequents,support,confidence,lift
1947,"frozenset({CodeRev_Yes, because I see value in code review, CareerSat_Very satisfied})",frozenset({JobSat_Very satisfied}),0.150161,0.626899,2.478835
1952,frozenset({JobSat_Very satisfied}),"frozenset({WelcomeChange_Just as welcome now as I felt last year, CareerSat_Very satisfied})",0.153428,0.606672,2.465174
1950,"frozenset({WelcomeChange_Just as welcome now as I felt last year, CareerSat_Very satisfied})",frozenset({JobSat_Very satisfied}),0.153428,0.623444,2.465174
1948,"frozenset({JobSat_Very satisfied, CodeRev_Yes, because I see value in code review})",frozenset({CareerSat_Very satisfied}),0.150161,0.809460,2.463314
1110,"frozenset({Hobbyist_Yes, JobSat_Very satisfied})",frozenset({CareerSat_Very satisfied}),0.163734,0.805452,2.451118
1467,"frozenset({Employment_Employed full-time, JobSat_Very satisfied})",frozenset({CareerSat_Very satisfied}),0.173748,0.801007,2.437591
1949,"frozenset({WelcomeChange_Just as welcome now as I felt last year, JobSat_Very satisfied})",frozenset({CareerSat_Very satisfied}),0.153428,0.800717,2.436708
1468,"frozenset({Employment_Employed full-time, CareerSat_Very satisfied})",frozenset({JobSat_Very satisfied}),0.173748,0.615498,2.433756
1470,frozenset({JobSat_Very satisfied}),"frozenset({Employment_Employed full-time, CareerSat_Very satisfied})",0.173748,0.687021,2.433756
214,frozenset({CareerSat_Very satisfied}),frozenset({JobSat_Very satisfied}),0.201514,0.613238,2.424820


In [8]:
def contiene_proxy_bigfive(frozenset_items):
    return any(item.startswith(('Neuroticismo_Nivel_', 'Amabilidad_Nivel_', 'Apertura_Nivel_')) for item in frozenset_items)

reglas_con_proxy = reglas_limpias[
    reglas_limpias.apply(lambda row: contiene_proxy_bigfive(row['antecedents'] | row['consequents']), axis=1)
].sort_values('lift', ascending=False)

print(f"Reglas limpias que sí involucran algún proxy de Big Five: {len(reglas_con_proxy)}")
display(reglas_con_proxy[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas limpias que sí involucran algún proxy de Big Five: 340


,antecedents,consequents,support,confidence,lift
4616,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Neuroticismo_Nivel_Medio})","frozenset({Employment_Employed full-time, WorkLoc_Office})",0.193133,0.657968,1.588133
5206,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), WelcomeChange_Just as welcome now as I felt last year, Neuroticismo_Nivel_Medio})",frozenset({WorkLoc_Office}),0.156886,0.703576,1.538779
4613,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Employment_Employed full-time, Neuroticismo_Nivel_Medio})",frozenset({WorkLoc_Office}),0.193133,0.700953,1.533040
5209,"frozenset({Neuroticismo_Nivel_Medio, WorkLoc_Office})","frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), WelcomeChange_Just as welcome now as I felt last year})",0.156886,0.693177,1.530094
4617,"frozenset({Neuroticismo_Nivel_Medio, WorkLoc_Office})","frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Employment_Employed full-time})",0.193133,0.853332,1.526992
2215,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Neuroticismo_Nivel_Medio})",frozenset({WorkLoc_Office}),0.204657,0.697226,1.524889
4615,"frozenset({Employment_Employed full-time, WorkLoc_Office, Neuroticismo_Nivel_Medio})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.193133,0.919850,1.518397
5208,"frozenset({WelcomeChange_Just as welcome now as I felt last year, WorkLoc_Office, Neuroticismo_Nivel_Medio})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.156886,0.917100,1.513858
3936,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Hobbyist_Yes, Neuroticismo_Nivel_Medio})",frozenset({WorkLoc_Office}),0.157426,0.685468,1.499173
4620,"frozenset({WorkLoc_Office, Employment_Employed full-time, Apertura_Nivel_Medio})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.152391,0.906891,1.497006


In [9]:
def contiene_extremo_bigfive(frozenset_items):
    return any(item.endswith('_Alto') or item.endswith('_Bajo') 
               for item in frozenset_items 
               if item.startswith(('Neuroticismo_Nivel_', 'Amabilidad_Nivel_', 'Apertura_Nivel_')))

reglas_extremos = reglas_limpias[
    reglas_limpias.apply(lambda row: contiene_extremo_bigfive(row['antecedents'] | row['consequents']), axis=1)
].sort_values('lift', ascending=False)

print(f"Reglas limpias con nivel Alto/Bajo de algún proxy: {len(reglas_extremos)}")
display(reglas_extremos[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas limpias con nivel Alto/Bajo de algún proxy: 87


,antecedents,consequents,support,confidence,lift
5033,"frozenset({WorkWeekHrs_Bin_35-45h, Amabilidad_Nivel_Bajo})","frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Employment_Employed full-time})",0.162844,0.799303,1.430310
5028,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Employment_Employed full-time, Amabilidad_Nivel_Bajo})",frozenset({WorkWeekHrs_Bin_35-45h}),0.162844,0.767397,1.407807
5030,"frozenset({Employment_Employed full-time, WorkWeekHrs_Bin_35-45h, Amabilidad_Nivel_Bajo})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.162844,0.849263,1.401879
5031,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Amabilidad_Nivel_Bajo})","frozenset({Employment_Employed full-time, WorkWeekHrs_Bin_35-45h})",0.162844,0.708642,1.399128
2319,"frozenset({Apertura_Nivel_Alto, WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})","frozenset({CodeRev_Yes, because I see value in code review})",0.156920,0.759969,1.393288
5032,"frozenset({Employment_Employed full-time, Amabilidad_Nivel_Bajo})","frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), WorkWeekHrs_Bin_35-45h})",0.162844,0.609486,1.367146
2456,"frozenset({WorkWeekHrs_Bin_35-45h, Amabilidad_Nivel_Bajo})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.168735,0.828219,1.367142
2455,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Amabilidad_Nivel_Bajo})",frozenset({WorkWeekHrs_Bin_35-45h}),0.168735,0.734278,1.347048
1932,"frozenset({Amabilidad_Nivel_Alto, WorkWeekHrs_Bin_35-45h})",frozenset({Employment_Employed full-time}),0.166460,0.970385,1.337612
5029,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), WorkWeekHrs_Bin_35-45h, Amabilidad_Nivel_Bajo})",frozenset({Employment_Employed full-time}),0.162844,0.965087,1.330308


### 3.2. Generación de Itemsets Frecuentes (Algoritmo FP-Growth)

Dado que contamos con una matriz de casi 90.000 filas y más de 200 columnas, el algoritmo clásico *Apriori* sería extremadamente costoso a nivel computacional al tener que generar múltiples combinaciones candidatas por iteración. 

Utilizaremos **FP-Growth** (Frequent Pattern Growth), que comprime el dataset en una estructura de árbol (FP-Tree). Definiremos un **Soporte Mínimo (`min_support`)**. 

> *Justificación de negocio:* Un soporte del **0.05 (5%)** significa que un conjunto de características debe aparecer en al menos ~4.400 candidatos para ser considerado un patrón representativo y no un caso aislado.

In [10]:
start = time.time()
frequent_itemsets_fpgrowth = fpgrowth(
    df_matriz,
    min_support=0.15,
    use_colnames=True,
    max_len=4
)
tiempo_fpgrowth = time.time() - start

print(f"Itemsets frecuentes (FP-Growth): {len(frequent_itemsets_fpgrowth)}")
print(f"Tiempo de ejecución: {tiempo_fpgrowth:.2f} segundos")

Itemsets frecuentes (FP-Growth): 2188
Tiempo de ejecución: 17.46 segundos


In [ ]:
reglas_fpgrowth = association_rules(
    frequent_itemsets_fpgrowth,
    metric="confidence",
    min_threshold=0.6
)

print(f"Itemsets — Apriori: {len(itemsets_frecuentes)} | FP-Growth: {len(frequent_itemsets_fpgrowth)}")
print(f"Reglas — Apriori: {len(reglas_asociacion)} | FP-Growth: {len(reglas_fpgrowth)}")

Itemsets — Apriori: 2188 | FP-Growth: 2188
Reglas    — Apriori: 5324 | FP-Growth: 5324


### 3.3. Generación de Reglas de Asociación

A partir de los Itemsets Frecuentes, extraemos las reglas direccionales (Ej: *Si [Antecedente] -> Entonces [Consecuente]*). 
Utilizaremos como métrica base la **Confianza (Confidence)**.

> *Justificación de negocio:* Una confianza de **0.50** indica que en al menos el 50% de los casos en los que se cumple la condición A (el perfil del candidato tiene ciertos rasgos), también se cumple la condición B.

In [11]:
MIN_CONFIDENCE = 0.50  # Mínimo 50% de probabilidad condicional

print("Generando reglas de asociación...")
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)

print(f"Se han generado {len(rules)} reglas de asociación base con confianza >= {MIN_CONFIDENCE * 100}%")
display(rules.head(5))

Generando reglas de asociación...


NameError: name 'frequent_itemsets' is not defined

### 3.4. Filtrado de Reglas por Relevancia (Lift)

Una regla puede tener alta confianza simplemente porque el consecuente es una característica que tiene casi todo el mundo (por ejemplo, "trabajar a tiempo completo"). Para separar las correlaciones genuinas del ruido estadístico, utilizamos la métrica **Lift**.

> *Justificación de negocio:* Un **Lift > 1** indica que la ocurrencia del Antecedente incrementa la probabilidad del Consecuente más allá de lo que dictaría el azar. Establecemos un filtro de **Lift >= 1.2** para asegurarnos de capturar reglas donde la asociación aporte valor real al descubrimiento de perfiles de NetSTAFF.

In [ ]:
MIN_LIFT = 1.2

# Filtrar por Lift
filtered_rules = rules[rules['lift'] >= MIN_LIFT].copy()

# Ordenar las reglas por Lift y Confianza para priorizar los patrones más determinantes
filtered_rules = filtered_rules.sort_values(by=['lift', 'confidence'], ascending=[False, False])

print(f"Tras filtrar por Lift >= {MIN_LIFT}, hemos extraído {len(filtered_rules)} reglas de alto valor.")
display(filtered_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

### 3.5. Exportación de la Base de Conocimiento

El DataFrame `filtered_rules` representa ahora el "cerebro" o la base empírica de nuestro framework en lo relativo a Soft Skills y comportamiento organizativo. Guardaremos estas reglas en formato CSV.

En fases posteriores (Inferencia y Matching), contrastaremos estas reglas estadísticas descubiertas por los datos frente a las reglas teóricas definidas por el experto en el documento de Riesgos.

In [ ]:
# Formatear frozensets a strings limpios para su exportación a CSV
filtered_rules['antecedents_str'] = filtered_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
filtered_rules['consequents_str'] = filtered_rules['consequents'].apply(lambda x: ', '.join(list(x)))

export_path = os.path.join(os.path.dirname(config.MATRIZ_TRANSACCIONAL_PATH), 'reglas_asociacion_soft.csv')
filtered_rules.to_csv(export_path, index=False)

print(f"Base de Conocimiento de Soft Skills exportada exitosamente a:\n{export_path}")